# Анализ матчей League of Legends на Kaggle-датасете

## Цель

Использовать крупный датасет матчей League of Legends для анализа чемпионов, игровых метрик и факторов победы.

API-пайплайн уже был реализован отдельно. В этом ноутбуке используем Kaggle-датасет как основной источник данных для анализа.

In [9]:
# pandas нужен для загрузки Excel-файла и работы с таблицами.
import pandas as pd


In [ ]:
# Загружаем Kaggle-датасет из Excel-файла.
# После уборки проекта исходный файл лежит в data/raw.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

league_df = pd.read_excel(PROJECT_ROOT / "data" / "raw" / "league_data.xlsx")
league_df.head()


In [11]:
# Проверяем размер таблицы: количество строк и столбцов.
print("Размер таблицы:", league_df.shape)

# Смотрим типы данных и количество пропусков по каждому столбцу.
league_df.info()

# Выводим список всех колонок, чтобы выбрать нужные для анализа.
league_df.columns.tolist()


(40410, 94)

In [12]:
# Добавляем производные метрики для дальнейшего анализа.

# KDA = (kills + assists) / deaths.
# Если смертей 0, заменяем deaths на 1, чтобы не делить на ноль.
league_df["kda"] = (
    (league_df["kills"] + league_df["assists"])
    / league_df["deaths"].clip(lower=1)
)

# Переводим длительность матча из секунд в минуты.
league_df["game_duration_min"] = league_df["game_duration"] / 60

# Считаем урон по чемпионам в минуту.
league_df["damage_per_min"] = (
    league_df["total_damage_dealt_to_champions"]
    / league_df["game_duration_min"]
)

# Считаем заработанное золото в минуту.
league_df["gold_per_min"] = (
    league_df["gold_earned"]
    / league_df["game_duration_min"]
)


,game_id,game_start_utc,game_duration,game_mode,game_type,game_version,map_id,platform_id,queue_id,participant_id,...,final_magicPen,final_magicPenPercent,final_magicResist,final_movementSpeed,final_omnivamp,final_physicalVamp,final_power,final_powerMax,final_powerRegen,final_spellVamp
0,3727443167,2025-01-15 14:56:00,1714,CLASSIC,MATCHED_GAME,15.1.649.4112,11,EUN1,420,5,...,0,0,48,385,0,0,799,1134,147,0
1,3726377460,2025-01-13 10:50:00,1300,CLASSIC,MATCHED_GAME,15.1.648.3927,11,EUN1,420,5,...,0,0,38,390,0,0,970,970,105,0
2,3729643655,2025-01-19 18:15:00,2019,CLASSIC,MATCHED_GAME,15.1.649.4112,11,EUN1,420,2,...,0,0,121,431,0,0,10000,10000,0,0
3,3729915593,2025-01-20 01:27:00,1625,CLASSIC,MATCHED_GAME,15.1.649.4112,11,EUN1,420,8,...,12,0,47,380,0,0,1122,1596,37,0
4,3729901665,2025-01-20 00:40:00,1542,CLASSIC,MATCHED_GAME,15.1.649.4112,11,EUN1,420,10,...,0,0,40,534,0,0,1025,1025,109,0


In [16]:
league_df[[
    "champion_name", "team_position", "kills", "deaths", "assists",
    "kda", "game_duration_min", "damage_per_min", "gold_per_min"
]].head(10)

,champion_name,team_position,kills,deaths,assists,kda,game_duration_min,damage_per_min,gold_per_min
0,Nami,UTILITY,0,3,22,7.333333,28.566667,346.522754,296.744457
1,Lulu,UTILITY,1,6,2,0.500000,21.666667,194.169231,242.353846
2,Viego,JUNGLE,11,11,11,2.000000,33.650000,817.236256,373.551263
3,Malzahar,MIDDLE,1,11,3,0.363636,27.083333,586.781538,342.498462
4,Lulu,UTILITY,1,4,3,1.000000,25.700000,190.544747,236.575875
5,Lulu,UTILITY,3,2,6,4.500000,25.650000,274.580897,263.391813
6,MonkeyKing,NaN,11,8,27,4.750000,12.350000,1597.327935,804.048583
7,Lulu,UTILITY,0,4,1,0.250000,23.650000,164.101480,210.486258
8,Aphelios,BOTTOM,0,7,5,0.714286,26.500000,758.490566,313.433962
9,Senna,UTILITY,1,8,16,2.125000,39.466667,559.155405,307.804054


In [17]:
league_df[[
    "kda",
    "game_duration_min",
    "damage_per_min",
    "gold_per_min"
]].describe()

,kda,game_duration_min,damage_per_min,gold_per_min
count,40410.000000,40410.000000,40410.000000,40410.000000
mean,3.453373,26.360932,906.413193,465.094547
std,3.373403,7.950860,553.740050,158.554463
min,0.000000,1.683333,0.000000,142.440945
25%,1.500000,21.283333,521.877483,344.862820
50%,2.571429,26.966667,790.219080,421.921333
75%,4.166667,31.383333,1160.159249,582.084947
max,40.000000,55.116667,5414.445118,1404.604317


In [18]:
print("Уникальных матчей:", league_df["game_id"].nunique())
print("Уникальных игроков:", league_df["puuid"].nunique())
print("Уникальных чемпионов:", league_df["champion_name"].nunique())

league_df["platform_id"].value_counts()

Уникальных матчей: 4044
Уникальных игроков: 28098
Уникальных чемпионов: 169


platform_id
EUN1    40410
Name: count, dtype: int64

In [19]:
league_df["solo_tier"].value_counts(dropna=False)

solo_tier
NaN            12544
DIAMOND         5941
GOLD            3949
MASTER          3472
EMERALD         3376
SILVER          3343
PLATINUM        2707
BRONZE          2400
IRON            1292
GRANDMASTER     1158
CHALLENGER       228
Name: count, dtype: int64

In [20]:
league_df["team_position"].value_counts(dropna=False)

team_position
NaN        9748
JUNGLE     6134
BOTTOM     6134
UTILITY    6132
TOP        6132
MIDDLE     6130
Name: count, dtype: int64